# Using the RooJSONFactoryWSTool to generate HS3 JSON with RooFit

This notebook shows how to export a `RooWorkspace` as an HS3 (Harmonized Statistics Serialization Standard) JSON file, and import it back into a fresh workspace.

The standard is still evolving. If you have questions or feedback about how you would like to use these workspaces in your analysis, please contact simon.cello@cern.ch.

# 0. Software setup

[HS3](https://hep-statistics-serialization-standard.github.io/) is a standardized format for publishing and archiving statistical models. It uses `JSON` to describe the computational graph of a likelihood while remaining human-readable.

With the experimental `RooJSONFactoryWSTool` in `ROOT`, a workspace can be translated to this format with relatively little code. Because the tool is still under active development, it is a good idea to work with a recent `ROOT` build.

Binder already provides that environment. On Google Colab, run the next cell once to install ROOT into the temporary runtime before continuing.


In [ ]:
import importlib
import importlib.util
import subprocess
import sys

try:
    import google.colab  # type: ignore  # noqa: F401
    in_colab = True
except ImportError:
    in_colab = False

has_root = importlib.util.find_spec("ROOT") is not None

if in_colab and not has_root:
    print("Installing ROOT for this Colab runtime. This can take a minute...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ROOT"])
    importlib.invalidate_caches()

import ROOT

# Print the ROOT build information .
print("ROOT Version =", ROOT.gROOT.GetVersion(), ",ROOT Date =", ROOT.gROOT.GetGitDate())


# 1. Create a simple RooFit workspace

RooFit provides a large toolbox for statistical model building. This tutorial keeps the physics model intentionally small so the focus stays on the export and import steps of `RooJSONFactoryWSTool`.

We will build a one-dimensional signal-plus-background model, generate toy data from it, and fit it once before exporting anything. That gives us a clear baseline for the round-trip test later on.


## 1.1 Define the observables and PDF components

The model below has a Gaussian signal component and an exponential background component. Both share the same observable `x`, and `sig_frac` controls their mixture in the final combined PDF.


In [2]:
# Define the observable that will be sampled and fitted.
x = ROOT.RooRealVar("x", "x", 0, 5)

# Signal model: a Gaussian peak with floating mean and width.
mean = ROOT.RooRealVar("mean", "mean", 2, 0, 5)
sigma = ROOT.RooRealVar("sigma", "sigma", 1, 1e-5, 2)
gauss = ROOT.RooGaussian("gauss", "gauss", x, mean, sigma)

# Background model: an exponential that falls across the observable range.
delta = ROOT.RooRealVar("delta", "delta", -1, -5, 0)
exponential = ROOT.RooExponential("exponential", "exponential", x, delta)

# Fraction of the signal component in the combined model.
sig_frac = ROOT.RooRealVar("sig_frac", "sig_frac", 0.6, 0, 1)

# Build a signal-plus-background PDF from the two components.
model = ROOT.RooAddPdf(
    "model",
    "model",
    ROOT.RooArgList(gauss, exponential),
    ROOT.RooArgList(sig_frac),
)


## 1.2 Generate a toy dataset

Once the PDF is defined, we can sample events from it. This gives us a synthetic dataset that behaves like a small analysis input and can later be stored in the workspace as well.


In [3]:
# Generate a toy dataset of 100000 events from the composite model.
data = model.generate(ROOT.RooArgSet(x), 100000)


## 1.3 Fit the model before exporting

Before writing anything to JSON, it is useful to fit the model once and inspect the result. This provides a reference point for checking that the imported workspace behaves the same way.


In [4]:
# Fit the model to the generated data and print the fitted parameters.
r = model.fitTo(data, ROOT.RooFit.Save(), PrintLevel=-1)
r.Print()


[#1] INFO:Fitting -- RooAbsPdf::fitTo(model) fixing normalization set for coefficient determination to observables in data
[#1] INFO:Fitting -- using generic CPU library compiled with no vectorizations
[#1] INFO:Fitting -- Creation of NLL object took 1.30388 ms
[#1] INFO:Fitting -- RooAddition::defaultErrorLevel(nll_model_modelData) Summation contains a RooNLLVar, using its error level
[#1] INFO:Minimization -- RooAbsMinimizerFcn::setOptimizeConst: activating const optimization
[#1] INFO:Minimization -- [fitFCN] No discrete parameters, performing continuous minimization only
[#1] INFO:Minimization -- RooAbsMinimizerFcn::setOptimizeConst: deactivating const optimization

  RooFitResult: minimized FCN value: 135610, estimated distance to minimum: 4.3782e-06
                covariance matrix quality: Full, accurate covariance matrix
                Status : MINIMIZE=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
          

## 2.1 Export the workspace to HS3 JSON

The `RooJSONFactoryWSTool` works on a `RooWorkspace`, so we first import the PDF and the dataset into a workspace container. The importer works itself recursively through the dependencies of the objects. 

The export step then writes an HS3 JSON representation to `tutorial.json` and stores the workspace in `tutorial.root`, so both artifacts are available in Binder.

In [ ]:
# Collect the model and dataset in a RooWorkspace before exporting.
ws = ROOT.RooWorkspace("WS")
ws.Import(model)
ws.Import(data)

# Export the workspace content to both HS3 JSON and a ROOT file.
tool = ROOT.RooJSONFactoryWSTool(ws)
tool.exportJSON("tutorial.json")
with ROOT.TFile.Open("tutorial.root", "RECREATE") as f:
    ws.Write()

## 2.2 Import the JSON into a fresh workspace

To test the round-trip, create a brand-new `RooWorkspace` and populate it from the JSON file. This mimics how a downstream user would reconstruct the model in a separate session.


In [ ]:
# Start from a fresh workspace and rebuild it from the JSON file.
ws_new = ROOT.RooWorkspace("WS")
tool_new = ROOT.RooJSONFactoryWSTool(ws_new)
tool_new.importJSON("tutorial.json")

# Retrieve the imported PDF and dataset by their workspace names.
model_new = ws_new.pdf("model")
data_new = ws_new.data("modelData")


[#1] INFO:ObjectHandling -- RooWorkspace::import() importing RooRealVar::delta
[#1] INFO:ObjectHandling -- RooWorkspace::import() importing RooRealVar::mean
[#1] INFO:ObjectHandling -- RooWorkspace::import() importing RooRealVar::sig_frac
[#1] INFO:ObjectHandling -- RooWorkspace::import() importing RooRealVar::sigma
[#1] INFO:ObjectHandling -- RooWorkspace::import() importing RooRealVar::x
[#1] INFO:ObjectHandling -- RooWorkspace::import() importing dataset modelData


## 2.3 Export an existing RooWorkspace from a ROOT file

You can also open the `tutorial.root` file written above and convert the stored workspace directly.

In [ ]:
# Reload the saved workspace from disk and export it again via the file-backed path.
with ROOT.TFile.Open("tutorial.root") as f:
    ws = f.Get("WS")
tool = ROOT.RooJSONFactoryWSTool(ws)
tool.exportJSON("tutorial.json")

## 2.4 Validate the round-trip

Now retrieve the imported PDF and dataset from the new workspace, fit them again, and compare the parameter values with the original fit. Small numerical differences are normal, but the overall result should be consistent.


In [ ]:
r_new = model_new.fitTo(data_new, ROOT.RooFit.Save(), PrintLevel=-1)
r_new.Print()
